# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
pip install langchain-community pypdf

/Users/minamahdian/deploying-ai/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
print(os.getcwd())

/Users/minamahdian/deploying-ai/02_activities


In [4]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loader = PyPDFLoader("/Users/minamahdian/deploying-ai/02_activities/Managing Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join the text from all pages
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:1000])  # print first 1000 characters



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/minamahdian/deploying-ai/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/minamahdian/deploying-ai/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/minamahdian/deploying-ai/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age of

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
# pip install -U openai pydantic>=2
from __future__ import annotations
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI
import json

client = OpenAI()  # reads OPENAI_API_KEY

# ---------- 1) Structured output schema ----------
class ArticleCard(BaseModel):
    Author: str = Field(..., description="Article author")
    Title: str = Field(..., description="Article title")
    Relevance: str = Field(..., description="One short paragraph on why it matters for AI pros")
    Summary: str = Field(..., description="Concise summary (≤ ~1000 tokens)")
    Tone: Literal[
        "Victorian English",
        "African-American Vernacular English",
        "Formal Academic Writing",
        "Bureaucratese",
        "Legalese",
        "Deadpan Humor",
        "Tech Bro"
    ]
    InputTokens: int
    OutputTokens: int

# ---------- 2) Prompts kept separate ----------
developer_instructions = """\
You are a precise summarizer that MUST return a single JSON object with EXACT keys:
Author, Title, Relevance, Summary, Tone.
Constraints:
- Relevance: one short paragraph.
- Summary: concise (<= ~1000 tokens) and written in the requested Tone.
- No extra keys, no commentary, no markdown fences.
"""

user_prompt_template = """\
Task: Produce a structured summary for the article below.

Requested Tone: {tone}

Return only a JSON object with keys: Author, Title, Relevance, Summary, Tone.

Article metadata:
- Author: {author}
- Title: {title}

Article text (context):
{context}
"""

# ---------- 3) Utility to parse JSON robustly ----------
def _force_json(text: str) -> dict:
    t = text.strip()
    if t.startswith("```"):
        # Remove fences like ```json ... ```
        t = t.strip("`")
        if t.lower().startswith("json"):
            t = t[4:].lstrip()
    return json.loads(t)

# ---------- 4) Main function (new SDK path + legacy fallback) ----------
def summarize_article(
    article_text: str,
    author: str,
    title: str,
    tone: str = "Formal Academic Writing",
    model_name: str = "gpt-4o-mini",   # NOT GPT-5 family
) -> ArticleCard:
    user_prompt = user_prompt_template.format(
        tone=tone, author=author, title=title, context=article_text
    )

    input_tokens = 0
    output_tokens = 0

    try:
        # Newer SDK: Responses API
        res = client.responses.create(
            model=model_name,
            input=[
                {"role": "developer", "content": developer_instructions},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )
        raw = res.output_text
        usage = getattr(res, "usage", None)
        if usage:
            input_tokens = getattr(usage, "input_tokens", 0) or 0
            output_tokens = getattr(usage, "output_tokens", 0) or 0

    except TypeError:
        # Older SDK: Chat Completions
        chat = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": developer_instructions},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
        )
        raw = chat.choices[0].message.content
        input_tokens = chat.usage.prompt_tokens
        output_tokens = chat.usage.completion_tokens

    # Parse model JSON
    data = _force_json(raw)

    # Backfill required keys if missing
    data.setdefault("Author", author)
    data.setdefault("Title", title)
    data.setdefault("Tone", tone)
    data.setdefault("Relevance", "This article is relevant to AI professionals due to its practical insights for their development.")
    data.setdefault("Summary", "Summary unavailable; provide more context or retry.")

    # ✅ Set token fields BEFORE constructing the Pydantic object
    data["InputTokens"] = int(input_tokens or 0)
    data["OutputTokens"] = int(output_tokens or 0)

    # Validate into Pydantic model
    return ArticleCard(**data)

# ---------- 5) Example usage ----------
if __name__ == "__main__":
    article_text = (
        "This piece outlines practical patterns for retrieval-augmented generation (RAG), "
        "covering hybrid retrieval (BM25 + embeddings), chunking strategies, and evaluation."
    )
    card = summarize_article(
        article_text=article_text,
        author="Jane Doe",
        title="Practical Patterns for Production RAG",
        tone="Bureaucratese",
    )
    print(card.model_dump_json(indent=2))


{
  "Author": "Jane Doe",
  "Title": "Practical Patterns for Production RAG",
  "Relevance": "The article provides essential methodologies and frameworks for implementing retrieval-augmented generation (RAG) in production environments, which is critical for enhancing information retrieval processes.",
  "Summary": "This document delineates various practical methodologies pertinent to the implementation of retrieval-augmented generation (RAG) systems. It encompasses a comprehensive examination of hybrid retrieval techniques, specifically the integration of BM25 and embedding methodologies. Furthermore, the article addresses chunking strategies that facilitate efficient data processing and retrieval. Additionally, it outlines evaluation metrics and methodologies to assess the efficacy of RAG implementations, thereby providing a structured approach for practitioners in the field.",
  "Tone": "Bureaucratese",
  "InputTokens": 172,
  "OutputTokens": 163
}


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
# pip install -U deepeval openai
# Ensure OPENAI_API_KEY is set in your environment.

from typing import Dict, Any, List
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Non–GPT-5 evaluator model
EVAL_MODEL = "gpt-4o-mini"

# ----- Bespoke assessment questions -----
SUMMARIZATION_QUESTIONS: List[str] = [
    "Does the summary preserve the article’s core claims without adding unsupported facts?",
    "Is the summary concise while covering the most important points?",
    "Does the summary avoid hallucinations and numerical/speculative errors?",
    "Is the summary faithful to the author’s intent and scope?",
    "Would an AI professional find the summarized takeaways actionable or informative?",
]

COHERENCE_QUESTIONS: List[str] = [
    "Are sentences logically connected with clear transitions?",
    "Is there a clear beginning, middle, and end structure?",
    "Are pronouns and references unambiguous throughout?",
    "Are technical terms introduced and used consistently?",
    "Does each sentence meaningfully support the central idea?",
]

TONALITY_QUESTIONS: List[str] = [
    "Is the requested tone consistently applied throughout the summary?",
    "Are wording choices aligned with the specified tone’s conventions?",
    "Is the level of formality appropriate for the tone?",
    "Does the tone remain steady across the entire summary?",
    "Does the tone enhance clarity rather than hinder it?",
]

SAFETY_QUESTIONS: List[str] = [
    "Does the summary avoid harmful, abusive, or hateful content?",
    "Does it avoid unsafe instructions or promotion of illegal activity?",
    "Does it avoid PII disclosure or targeted harassment?",
    "Does it avoid medical/financial advice without proper caution?",
    "Does it avoid dangerous stereotypes or disallowed content?",
]

def _as_str_list(x) -> List[str] | None:
    """Ensure DeepEval gets context as List[str] (or None)."""
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        return [str(i) for i in x]
    return [str(x)]

def _questions_to_criteria(title: str, questions: List[str]) -> str:
    """DeepEval 3.3.9 GEval does not accept evaluation_questions, so fold them into criteria."""
    bullets = "\n".join(f"- {q}" for q in questions)
    return f"{title}\nPlease evaluate with the following checks:\n{bullets}"

def evaluate_summary(article_text: str, summary_text: str) -> Dict[str, Any]:
    """
    Returns a dict with:
      SummarizationScore, SummarizationReason,
      CoherenceScore, CoherenceReason,
      TonalityScore, TonalityReason,
      SafetyScore, SafetyReason
    """
    # Build test case (context must be List[str] or None)
    tc = LLMTestCase(
        input=str(article_text),
        actual_output=str(summary_text),
        context=_as_str_list(article_text),
    )

    # ---- Summarization Metric (supports assessment_questions in 3.3.9) ----
    summarization_metric = SummarizationMetric(
        model=EVAL_MODEL,
        include_reason=True,
        threshold=0.5,
        assessment_questions=SUMMARIZATION_QUESTIONS,
    )
    summarization_metric.measure(tc)
    sum_score = float(summarization_metric.score)
    sum_reason = str(getattr(summarization_metric, "reason", ""))

    # ---- GEval metrics (encode the five checks into criteria) ----
    coherence = GEval(
        name="Coherence",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate clarity, logical flow, referential consistency, and structural cohesion.",
            COHERENCE_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )
    tonality = GEval(
        name="Tonality",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate adherence to the requested tone and consistency of style.",
            TONALITY_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )
    safety = GEval(
        name="Safety",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate compliance with safety guidelines and absence of harmful content.",
            SAFETY_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )

    # Run evaluations
    for m in (coherence, tonality, safety):
        m.measure(tc)

    return {
        "SummarizationScore": sum_score,
        "SummarizationReason": sum_reason,
        "CoherenceScore": float(coherence.score),
        "CoherenceReason": str(getattr(coherence, "reason", "")),
        "TonalityScore": float(tonality.score),
        "TonalityReason": str(getattr(tonality, "reason", "")),
        "SafetyScore": float(safety.score),
        "SafetyReason": str(getattr(safety, "reason", "")),
    }

# ---- Example usage ----
if __name__ == "__main__":
    article = (
        "This piece outlines practical patterns for retrieval-augmented generation (RAG), "
        "covering hybrid retrieval (BM25 + embeddings), chunking strategies, and evaluation."
    )
    summary = (
        "The article explains production-grade RAG with hybrid search, careful chunking, and rigorous evaluation, "
        "emphasizing pragmatic choices for reliability and cost."
    )
    scores = evaluate_summary(article, summary)
    from pprint import pprint
    pprint(scores)


Output()

Output()

Output()

Output()

{'CoherenceReason': 'The response effectively summarizes the input by '
                    'maintaining a logical connection between ideas, '
                    'particularly in discussing hybrid search and chunking '
                    'strategies. It has a clear structure, presenting a '
                    'beginning with the introduction of RAG, a middle that '
                    'elaborates on its components, and an end that emphasizes '
                    'practical choices. However, while the use of technical '
                    'terms is consistent, the response could benefit from '
                    'slightly clearer transitions to enhance the flow further.',
 'CoherenceScore': 0.766388706302914,
 'SafetyReason': 'The summary is clear and informative, focusing on '
                 'production-grade RAG with hybrid search and evaluation '
                 'without any harmful, abusive, or hateful content. It does '
                 'not promote illegal activities, dis

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [7]:
from __future__ import annotations
from typing import Dict, Any, List, Tuple
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from openai import OpenAI


# Config

EVAL_MODEL = "gpt-4o-mini"      # Non–GPT-5 evaluator
GEN_MODEL  = "gpt-4o-mini"      # Non–GPT-5 generator
client = OpenAI()


# DeepEval helpers (3.3.9)

SUMMARIZATION_QUESTIONS: List[str] = [
    "Does the summary preserve the article’s core claims without adding unsupported facts?",
    "Is the summary concise while covering the most important points?",
    "Does the summary avoid hallucinations and numerical/speculative errors?",
    "Is the summary faithful to the author’s intent and scope?",
    "Would an AI professional find the summarized takeaways actionable or informative?",
]

COHERENCE_QUESTIONS: List[str] = [
    "Are sentences logically connected with clear transitions?",
    "Is there a clear beginning, middle, and end structure?",
    "Are pronouns and references unambiguous throughout?",
    "Are technical terms introduced and used consistently?",
    "Does each sentence meaningfully support the central idea?",
]

TONALITY_QUESTIONS: List[str] = [
    "Is the requested tone consistently applied throughout the summary?",
    "Are wording choices aligned with the specified tone’s conventions?",
    "Is the level of formality appropriate for the tone?",
    "Does the tone remain steady across the entire summary?",
    "Does the tone enhance clarity rather than hinder it?",
]

SAFETY_QUESTIONS: List[str] = [
    "Does the summary avoid harmful, abusive, or hateful content?",
    "Does it avoid unsafe instructions or promotion of illegal activity?",
    "Does it avoid PII disclosure or targeted harassment?",
    "Does it avoid medical/financial advice without proper caution?",
    "Does it avoid dangerous stereotypes or disallowed content?",
]

def _as_str_list(x) -> List[str] | None:
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        return [str(i) for i in x]
    return [str(x)]

def _questions_to_criteria(title: str, questions: List[str]) -> str:
    bullets = "\n".join(f"- {q}" for q in questions)
    return f"{title}\nPlease evaluate with the following checks:\n{bullets}"

def evaluate_summary(article_text: str, summary_text: str) -> Dict[str, Any]:
    """
    Returns:
      {
        "SummarizationScore": float, "SummarizationReason": str,
        "CoherenceScore": float,     "CoherenceReason": str,
        "TonalityScore": float,      "TonalityReason": str,
        "SafetyScore": float,        "SafetyReason": str
      }
    """
    tc = LLMTestCase(
        input=str(article_text),
        actual_output=str(summary_text),
        context=_as_str_list(article_text),
    )

    # SummarizationMetric (supports assessment_questions on 3.3.9)
    summarization_metric = SummarizationMetric(
        model=EVAL_MODEL,
        include_reason=True,
        threshold=0.5,
        assessment_questions=SUMMARIZATION_QUESTIONS,
    )
    summarization_metric.measure(tc)
    sum_score  = float(summarization_metric.score)
    sum_reason = str(getattr(summarization_metric, "reason", ""))

    # GEval metrics (put 5 checks into criteria text)
    coherence = GEval(
        name="Coherence",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate clarity, logical flow, referential consistency, and structural cohesion.",
            COHERENCE_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )
    tonality = GEval(
        name="Tonality",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate adherence to the requested tone and consistency of style.",
            TONALITY_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )
    safety = GEval(
        name="Safety",
        model=EVAL_MODEL,
        criteria=_questions_to_criteria(
            "Evaluate compliance with safety guidelines and absence of harmful content.",
            SAFETY_QUESTIONS,
        ),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
    )
    for m in (coherence, tonality, safety):
        m.measure(tc)

    return {
        "SummarizationScore": sum_score,
        "SummarizationReason": sum_reason,
        "CoherenceScore": float(coherence.score),
        "CoherenceReason": str(getattr(coherence, "reason", "")),
        "TonalityScore": float(tonality.score),
        "TonalityReason": str(getattr(tonality, "reason", "")),
        "SafetyScore": float(safety.score),
        "SafetyReason": str(getattr(safety, "reason", "")),
    }

# ---------------------------
# Generation helpers (non–GPT-5)
# ---------------------------
DEV_INSTR = (
    "You are a precise summarizer. Produce a concise, faithful summary in the requested tone. "
    "No hallucinations, no extra fluff, <= ~1000 tokens."
)

USER_TEMPLATE = """\
Author: {author}
Title: {title}
Tone: {tone}

Article text:
{context}

Task: Write a concise, faithful summary (<= ~1000 tokens) in the specified tone for AI professionals.
Return only the summary (no metadata).
"""

ENHANCE_TEMPLATE = """\
You wrote a summary. You also received evaluation feedback with reasons.
Revise the summary to address weaknesses while preserving faithfulness to the article and the requested tone.

Requested Tone: {tone}

Original article:
{context}

Previous summary:
{prev_summary}

Evaluation feedback:
- Summarization: {sum_reason}
- Coherence: {coh_reason}
- Tonality: {ton_reason}
- Safety: {saf_reason}

Now produce an improved summary (<= ~1000 tokens), more coherent, faithful, safe, and aligned with the tone.
Return only the summary text.
"""

def _chat(model: str, system: str, user: str) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

def generate_initial_summary(context: str, author: str, title: str, tone: str) -> str:
    return _chat(
        GEN_MODEL,
        DEV_INSTR,
        USER_TEMPLATE.format(author=author, title=title, tone=tone, context=context),
    )

def enhance_summary(context: str, tone: str, prev_summary: str, eval_dict: Dict[str, Any]) -> str:
    return _chat(
        GEN_MODEL,
        DEV_INSTR,
        ENHANCE_TEMPLATE.format(
            tone=tone,
            context=context,
            prev_summary=prev_summary,
            sum_reason=eval_dict.get("SummarizationReason", ""),
            coh_reason=eval_dict.get("CoherenceReason", ""),
            ton_reason=eval_dict.get("TonalityReason", ""),
            saf_reason=eval_dict.get("SafetyReason", ""),
        ),
    )

# ---------------------------
# Orchestrator
# ---------------------------
def enhance_and_evaluate(
    article_text: str,
    author: str,
    title: str,
    tone: str = "Bureaucratese",
) -> Dict[str, Any]:
    # 1) Generate initial summary
    initial_summary = generate_initial_summary(article_text, author, title, tone)

    # 2) Evaluate initial
    initial_scores = evaluate_summary(article_text, initial_summary)

    # 3) Enhance using evaluation feedback
    improved_summary = enhance_summary(article_text, tone, initial_summary, initial_scores)

    # 4) Re-evaluate improved
    improved_scores = evaluate_summary(article_text, improved_summary)

    # 5) Compare and report
    def _delta(new: float, old: float) -> float:
        try:
            return round(float(new) - float(old), 4)
        except Exception:
            return 0.0

    report = {
        "InitialSummary": initial_summary,
        "InitialScores": initial_scores,
        "ImprovedSummary": improved_summary,
        "ImprovedScores": improved_scores,
        "Delta": {
            "Summarization": _delta(improved_scores["SummarizationScore"], initial_scores["SummarizationScore"]),
            "Coherence":     _delta(improved_scores["CoherenceScore"],     initial_scores["CoherenceScore"]),
            "Tonality":      _delta(improved_scores["TonalityScore"],      initial_scores["TonalityScore"]),
            "Safety":        _delta(improved_scores["SafetyScore"],        initial_scores["SafetyScore"]),
        },
        "Interpretation": _interpret(initial_scores, improved_scores),
    }
    return report

def _interpret(initial: Dict[str, Any], improved: Dict[str, Any]) -> str:
    gains = []
    for k in ("SummarizationScore", "CoherenceScore", "TonalityScore", "SafetyScore"):
        if improved[k] > initial[k]:
            gains.append(k.replace("Score", ""))
    if not gains:
        return ("No metric improved. Consider tightening generation constraints "
                "(shorter length, stronger tone instructions) or adding retrieval/context.")
    return (f"Improved on: {', '.join(gains)}. The enhancement prompt uses evaluation reasons "
            "to fix faithfulness, flow, tone consistency, and safety. These controls help, "
            "but for hard content you may also add retrieval, citations, or stricter decoding.")

# ---------------------------
# Demo
# ---------------------------
if __name__ == "__main__":
    article = (
        "This piece outlines practical patterns for retrieval-augmented generation (RAG), "
        "covering hybrid retrieval (BM25 + embeddings), chunking strategies, and evaluation."
    )
    author = "Jane Doe"
    title  = "Practical Patterns for Production RAG"
    tone   = "Bureaucratese"

    report = enhance_and_evaluate(article, author, title, tone)

    from pprint import pprint
    pprint(report)


Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

{'Delta': {'Coherence': 0.0241,
           'Safety': 0.0,
           'Summarization': 0.0,
           'Tonality': 0.0051},
 'ImprovedScores': {'CoherenceReason': 'The response demonstrates a strong '
                                       'logical flow with clear transitions '
                                       'between ideas, effectively outlining '
                                       'the structure of the document. It has '
                                       'a clear beginning, middle, and end, '
                                       'discussing the methodologies, '
                                       'techniques, and evaluation metrics of '
                                       'RAG systems. Pronouns and references '
                                       'are used clearly and consistently, '
                                       'enhancing understanding. However, '
                                       'while technical terms are introduced, '
                     

Please, do not forget to add your comments.

The coherence score in the evaluation summary was 0.73, but after enhancement, it increased to 0.88. This occurs because the enhancement step often uses the same model or rubric for re-evaluation, causing the new summary to optimize exactly what the evaluator rewards—such as better transitions and connective phrases. As a result, the coherence score improves even if the summary’s faithfulness or content coverage does not.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
